# DFU Repair-7 v1.3 — Preserve 38 Good Trials
Uses the completed V4 reproducibility PKL to restore one missing good per-trial evidence identity without retraining, then repairs only the original seven incompatible fold-1 trials.


In [ ]:
import base64, gzip, hashlib, json, urllib.request

BASE_URL='https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/661fc29435c208971b8fb332ff804b672740dc65/notebooks/repair7_v1_2_payload/DFU_Repair7_Only_Preserve38_v1_2.ipynb.gz.b64'
BASE_SHA='f705473a0d8b9b73a3525bc94cbefc9fe4736ea60b46c4a1fbf1e39ea68fd6bc'
encoded=urllib.request.urlopen(BASE_URL, timeout=120).read().strip()
nb_bytes=gzip.decompress(base64.b64decode(encoded, validate=True))
actual=hashlib.sha256(nb_bytes).hexdigest()
if actual != BASE_SHA:
    raise RuntimeError(f"Repair-7 v1.2 base SHA mismatch: {actual} != {BASE_SHA}")
nb=json.loads(nb_bytes.decode("utf-8"))
code_cells=[c for c in nb["cells"] if c.get("cell_type")=="code"]
if len(code_cells)!=1:
    raise RuntimeError(f"Expected one base code cell, found {len(code_cells)}")
script="".join(code_cells[0]["source"])
marker='if sum(len(v) for v in expected_by_fold.values()) != 1055:\n    raise RuntimeError("Locked fold partition does not cover exactly 1055 images.")\n'
patch='\n# v1.3 fallback: restore the missing GOOD identity from the completed V4 reproducibility PKL.\n# No training is allowed here. The PKL evidence must exactly match the locked fold-1 image IDs.\n_v13_identity = ("mobilenetv3_large", 2028, 0)\n_v13_pkl = RUN_ROOT / "reliable_dfu_reproducibility_v4.pkl"\n_v13_need_restore = True\nif AGG_PRED_PATH.is_file() and AGG_METRIC_PATH.is_file():\n    try:\n        _ap = pd.read_csv(AGG_PRED_PATH)\n        _am = pd.read_csv(AGG_METRIC_PATH)\n        _pp = _ap[( _ap.model_key.astype(str)==_v13_identity[0]) & (_ap.seed.astype(int)==_v13_identity[1]) & (_ap.outer_fold.astype(int)==1)]\n        _mm = _am[( _am.model_key.astype(str)==_v13_identity[0]) & (_am.seed.astype(int)==_v13_identity[1]) & (_am.outer_fold.astype(int)==1)]\n        _v13_need_restore = not (len(_pp)==209 and _pp.image_id.astype(str).nunique()==209 and len(_mm)==1)\n    except Exception:\n        _v13_need_restore = True\nif _v13_need_restore:\n    if not _v13_pkl.is_file():\n        raise RuntimeError(f"v1.3 cannot recover the missing good trial: reproducibility PKL not found: {_v13_pkl}")\n    with _v13_pkl.open("rb") as _fh:\n        _v13_obj = pickle.load(_fh)\n    _v13_pred_all = pd.DataFrame(_v13_obj.get("predictions", []))\n    _v13_met_all = pd.DataFrame(_v13_obj.get("metrics", []))\n    _v13_p = _v13_pred_all[( _v13_pred_all.model_key.astype(str)==_v13_identity[0]) & (_v13_pred_all.seed.astype(int)==_v13_identity[1]) & (_v13_pred_all.outer_fold.astype(int)==1)].copy()\n    _v13_m = _v13_met_all[( _v13_met_all.model_key.astype(str)==_v13_identity[0]) & (_v13_met_all.seed.astype(int)==_v13_identity[1]) & (_v13_met_all.outer_fold.astype(int)==1)].copy()\n    _v13_expected = expected_by_fold[0]\n    _v13_ids = set(_v13_p.image_id.astype(str)) if not _v13_p.empty else set()\n    if len(_v13_p)!=209 or _v13_p.image_id.astype(str).duplicated().any() or _v13_ids != _v13_expected or len(_v13_m)!=1:\n        raise RuntimeError(\n            "v1.3 PKL recovery evidence failed locked-split verification: "\n            f"pred_rows={len(_v13_p)}, unique={len(_v13_ids)}, metric_rows={len(_v13_m)}, "\n            f"unexpected={len(_v13_ids-_v13_expected)}, missing={len(_v13_expected-_v13_ids)}"\n        )\n    _base_p = pd.read_csv(AGG_PRED_PATH) if AGG_PRED_PATH.is_file() else pd.DataFrame(columns=_v13_p.columns)\n    if not _base_p.empty:\n        _base_p = _base_p[~(( _base_p.model_key.astype(str)==_v13_identity[0]) & (_base_p.seed.astype(int)==_v13_identity[1]) & (_base_p.outer_fold.astype(int)==1))]\n    _base_m = pd.read_csv(AGG_METRIC_PATH) if AGG_METRIC_PATH.is_file() else pd.DataFrame(columns=_v13_m.columns)\n    if not _base_m.empty:\n        _base_m = _base_m[~(( _base_m.model_key.astype(str)==_v13_identity[0]) & (_base_m.seed.astype(int)==_v13_identity[1]) & (_base_m.outer_fold.astype(int)==1))]\n    AGG_PRED_PATH.parent.mkdir(parents=True, exist_ok=True)\n    pd.concat([_base_p, _v13_p], ignore_index=True).to_csv(AGG_PRED_PATH, index=False)\n    pd.concat([_base_m, _v13_m], ignore_index=True).to_csv(AGG_METRIC_PATH, index=False)\n    print("v1.3 PKL evidence restored to aggregate (NO TRAINING): mobilenetv3_large seed=2028 fold=1 | rows=209")\n'
if script.count(marker)!=1:
    raise RuntimeError(f"v1.3 patch marker count is {script.count(marker)}, expected 1")
script=script.replace(marker, marker+"\n"+patch, 1)
script=script.replace("DFU REPAIR-7 v1.2 ONLY", "DFU REPAIR-7 v1.3 ONLY", 1)
print("Pinned Repair-7 v1.3 patch verification: PASS")
exec(compile(script, "DFU_Repair7_Only_Preserve38_v1_3.py", "exec"), globals())
